# Semantic Scholar vs OpenAlex — citation coverage

The exploration behind [`docs/citation-coverage.md`](../../docs/citation-coverage.md)
and the recurring question **"could we drop Semantic Scholar (S2) and make
OpenAlex (OA) the single source of truth?"** Three experiments, against the
**live** APIs (through the app's throttled, keyed clients):

1. **Citation-count coverage** — does OA's `cited_by_count` match S2's
   `citationCount`, across ML and physics seeds?
2. **Resolution & the preprint→published gap** — the app resolves a seed to OA
   via the arXiv DOI; does that land the citation-rich published record or a
   preprint stub, and can OA even reach the published one?
3. **Top-cited citer overlap** — the app only keeps the top-`cite_limit` citers
   by impact, so the real question isn't *how many* citations OA has but whether
   it surfaces the *same landmark citers* S2 does.

> **Rate limits.** These hit S2 and OA live. S2 throttles ~1 req/s even keyed and
> 429s on deep citation paging. The result tables below are **recorded from a run
> on 2026-07-12**; the code is runnable but re-execution may be slow / throttled
> (that's why cells ship without pre-computed outputs). Counts drift over time —
> the *patterns* are the point.

## Setup

Helpers over the app's own integration clients (a couple of private internals — `_try_entity`, `_clean_search` — are used deliberately; this is exploratory, not app code).

In [ ]:
import re
import time
import urllib.parse

from atlas.config import config
from atlas.integrations.openalex import client as oaclient
from atlas.integrations.openalex import nodes as oanodes
from atlas.integrations.openalex import traversal as oatrav
from atlas.integrations.semantic_scholar import client as s2client

S2_GRAPH = config.providers.s2.graph_url
SELECT = "id,title,publication_year,cited_by_count,type"


def s2_batch(ids, fields):
    """One POST /paper/batch (the 429-safe bulk endpoint), raw rows."""
    url = f"{S2_GRAPH}/paper/batch?fields={urllib.parse.quote(fields)}"
    data = s2client.request(url, method="POST", body={"ids": list(ids)})
    return data if isinstance(data, list) else []


def oa_by_doi(arxiv_id):
    """OA record via the arXiv DOI — i.e. the *preprint* work."""
    return oatrav._try_entity(f"doi:10.48550/arXiv.{arxiv_id}", SELECT) or {}


def oa_by_title(title, article_only=False):
    """OA's most-cited same-title record (optionally constrained to articles)."""
    cleaned = oatrav._clean_search(title or "")
    if not cleaned:
        return {}
    filt = f"title.search:{cleaned}" + (",type:article" if article_only else "")
    params = {"filter": filt, "sort": "cited_by_count:desc", "per-page": "1", "select": SELECT}
    data = oaclient.request(oaclient.works_url(params))
    results = (data.get("results") if isinstance(data, dict) else None) or []
    return results[0] if results else {}


def norm(title):
    """Normalize a title for cross-source matching."""
    return re.sub(r"[^a-z0-9]", "", (title or "").lower())[:45]

## Experiment 1 — Citation-count coverage (ML vs physics)

For each arXiv seed, compare **S2 `citationCount`** with **OpenAlex's best
count** (the higher of its preprint record and its most-cited same-title
record). Well-known seeds only, so *resolution* isn't the confound — the
question is raw coverage, and whether it differs by field.

In [ ]:
import statistics

# (arxiv_id, label, field)
SEEDS = [
    ("1512.03385", "ResNet", "ml"), ("1706.03762", "Transformer", "ml"),
    ("1810.04805", "BERT", "ml"), ("1409.1556", "VGG", "ml"),
    ("1502.03167", "BatchNorm", "ml"), ("1412.6980", "Adam", "ml"),
    ("1406.2661", "GAN", "ml"), ("1707.06347", "PPO", "ml"),
    ("1609.02907", "GCN", "ml"), ("2005.14165", "GPT-3", "ml"),
    ("2010.11929", "ViT", "ml"), ("1301.3781", "word2vec", "ml"),
    ("1905.11946", "EfficientNet", "ml"), ("1907.11692", "RoBERTa", "ml"),
    ("2103.00020", "CLIP", "ml"), ("1803.11485", "QMIX", "ml"),
    ("2307.09288", "Llama 2", "ml"), ("1503.02531", "Distillation", "ml"),
    ("1602.03837", "LIGO GW150914", "phys"), ("hep-th/9711200", "Maldacena", "phys"),
    ("1207.7214", "ATLAS Higgs", "phys"), ("1207.7235", "CMS Higgs", "phys"),
    ("hep-ph/9905221", "Randall-Sundrum", "phys"),
]

s2_rows = s2_batch([f"ARXIV:{arxiv_id}" for arxiv_id, _, _ in SEEDS], "title,year,citationCount")
s2_by_seed = {arxiv_id: (row or {}) for (arxiv_id, _, _), row in zip(SEEDS, s2_rows)}

def fmt(value):
    """Render a possibly-missing count for the aligned table."""
    return "-" if value is None else str(value)


ratios = {"ml": [], "phys": []}
print(f"{'seed':<18}{'fld':>5}{'S2':>9}{'OA_doi':>8}{'OA_title':>9}{'best/S2':>9}")
for arxiv_id, label, field in SEEDS:
    s2c = s2_by_seed[arxiv_id].get("citationCount")
    title = s2_by_seed[arxiv_id].get("title")
    doi_c = oa_by_doi(arxiv_id).get("cited_by_count")
    ttl_c = oa_by_title(title).get("cited_by_count")
    best = max([count for count in (doi_c, ttl_c) if count], default=None)
    ratio = best / s2c if (isinstance(s2c, int) and s2c and isinstance(best, int)) else None
    if ratio is not None:
        ratios[field].append(ratio)
    ratio_text = f"{ratio:.2f}" if ratio is not None else "-"
    print(f"{label:<18}{field:>5}{fmt(s2c):>9}{fmt(doi_c):>8}{fmt(ttl_c):>9}{ratio_text:>9}")

print()
for field in ("ml", "phys"):
    values = ratios[field]
    under2 = sum(1 for ratio in values if ratio < 0.5)
    print(f"{field.upper():5} n={len(values):2} median OA_best/S2="
          f"{statistics.median(values):.2f} #(>2x undercount)={under2}/{len(values)}")

### Result (2026-07-12)

| Field | n | median OA_best / S2 | # undercounting > 2× |
|---|--:|--:|--:|
| **ML** | 17 | **0.23** | **10 / 17** |
| **Physics** | 5 | **1.19** | 0 / 5 |

The ML aggregate carries some *resolution noise* (generic titles like "BERT",
"Attention is all you need" mis-match a title search). The **clean signal** is
OA's own arXiv-DOI record (same paper, no resolution ambiguity) vs S2:
GPT-3 **0.05**, CLIP **0.10**, Llama 2 **0.15**, QMIX **0.17**, EfficientNet
0.20, GCN 0.22, ViT 0.33 … **median ≈ 0.28**. Physics is all ≥ 0.73, usually
**> 1**. So OA knows ~a quarter-to-a-third of the citers S2 does for
arXiv-native ML, and is fine-to-better for physics.

## Experiment 2 — Resolution & the preprint→published gap

The app resolves a seed to OA **arXiv-DOI first**. For papers with a real
published version (VoR), that DOI points at the low-cited **preprint** record.
Does a title search reach the published one, and does OA link the two?

In [ ]:
# DOI (preprint) vs title (most-cited) vs title+type:article, for a few seeds.
PROBE = ["1512.03385", "1602.03837", "hep-th/9711200", "2005.14165", "2307.09288"]
titles = {arxiv_id: (row or {}).get("title") for arxiv_id, row in
          zip(PROBE, s2_batch([f"ARXIV:{arxiv_id}" for arxiv_id in PROBE], "title"))}
print(f"{'arxiv':<16}{'OA_doi':>8}{'OA_title':>9}{'OA_article':>11}  best_type/year")
for arxiv_id in PROBE:
    doi = oa_by_doi(arxiv_id)
    ttl = oa_by_title(titles[arxiv_id])
    art = oa_by_title(titles[arxiv_id], article_only=True)
    cand = [(rec.get("cited_by_count"), rec) for rec in (doi, ttl, art) if rec.get("cited_by_count")]
    _, best = max(cand, default=(None, {}))
    print(f"{arxiv_id:<16}{fmt(doi.get('cited_by_count')):>8}"
          f"{fmt(ttl.get('cited_by_count')):>9}{fmt(art.get('cited_by_count')):>11}"
          f"  {best.get('type','-')}/{best.get('publication_year','-')}")

# Does the preprint record link to its published version?
print("\\nVersion-link probe (does a preprint point at its VoR?):")
for label, doi in [("ResNet", "10.48550/arXiv.1512.03385"),
                   ("GPT-3", "10.48550/arXiv.2005.14165")]:
    work = oatrav._try_entity("doi:" + doi, "id,type,cited_by_count,locations,related_works")
    if not work:
        continue
    srcs = [((loc.get("source") or {}).get("display_name")) for loc in work.get("locations") or []]
    print(f"  {label}: type={work.get('type')} cites={work.get('cited_by_count')} "
          f"locations={srcs} related_works={len(work.get('related_works') or [])}")

### Result (2026-07-12)

| Seed | OA arXiv-DOI (preprint) | OA title (canonical) |
|---|--:|--:|
| ResNet | 4,734 | **222,747** |
| LIGO GW150914 | 8 | **14,324** |
| Maldacena | 17 | **14,567** |
| GPT-3 | 3,029 | 3,029 |
| Llama 2 | 2,635 | 2,635 |

- **arXiv-DOI lands on the preprint stub** for papers with a VoR (ResNet, LIGO,
  Maldacena) — the app's current path, masked today only because `build.py`
  takes `max(S2, OA)`.
- **GPT-3 / Llama 2** are arXiv-only (title = DOI), so their low counts are a
  pure *coverage* gap, not resolution.
- **No preprint→VoR link.** The preprint record's `related_works` is empty and
  its `locations` list only arXiv/Zenodo mirrors — never the published article
  (a *separate* OA work). A `type:article` filter helps physics but *hurts*
  arXiv-only ML (forces a wrong/low-cited article match). So OA-only would still
  need a title/year canonical-picking heuristic.

## Experiment 3 — Top-cited citer overlap (quality over quantity)

`build.py` keeps only the top-`cite_limit` citers by impact, so what matters is
whether OA surfaces the **same landmark citers** as S2. Overlap of the top-15
most-cited citers, on RL papers small enough (< ~9k citers) that a full S2 pull
actually contains the true top-cited (S2 has no server-side citation sort and
caps paging at ~10k).

In [ ]:
S2_MAX_OFFSET = 8000  # offset + limit must stay <= 10000 (S2 hard cap)
TOP_N = 15
OVERLAP_SEEDS = [("1803.11485", "QMIX"), ("1706.02275", "MADDPG"), ("1710.02298", "Rainbow")]


def s2_top_citers(arxiv_id):
    """Page all S2 citers (title + count), sort by count desc, take TOP_N."""
    fields = urllib.parse.quote("title,citationCount")
    rows = []
    for offset in range(0, S2_MAX_OFFSET + 1, 1000):
        url = (f"{S2_GRAPH}/paper/ARXIV:{arxiv_id}/citations"
               f"?fields={fields}&limit=1000&offset={offset}")
        try:
            data = s2client.request(url)
        except Exception as error:  # noqa: BLE001 - one-off probe
            print(f"    (S2 pull stopped at offset {offset}: {error})")
            break
        items = data.get("data") if isinstance(data, dict) else None
        if not items:
            break
        for item in items:
            paper = (item or {}).get("citingPaper") or {}
            rows.append((paper.get("title") or "", paper.get("citationCount") or 0))
        if len(items) < 1000:
            break
        time.sleep(1.5)
    rows.sort(key=lambda pair: pair[1], reverse=True)
    return rows[:TOP_N]


def oa_top_citers(title):
    """Resolve the seed's most-cited OA record, then its top citers by cited_by."""
    seed = oa_by_title(title)
    bare = oanodes.bare_openalex_id(seed.get("id")) if seed else None
    if not bare:
        return []
    return [((edge.get("node") or {}).get("title") or "",
             (edge.get("node") or {}).get("citation_count") or 0)
            for edge in oatrav.citations(bare, TOP_N)]


for arxiv_id, label in OVERLAP_SEEDS:
    title = (s2_batch([f"ARXIV:{arxiv_id}"], "title")[0] or {}).get("title")
    s2_top = s2_top_citers(arxiv_id)
    oa_top = oa_top_citers(title)
    oa_keys = {norm(citer_title) for citer_title, _ in oa_top}
    overlap = sum(1 for citer_title, _ in s2_top if norm(citer_title) in oa_keys)
    print(f"\\n=== {label} ({arxiv_id}) — top-{TOP_N} citer overlap: {overlap}/{TOP_N} ===")
    for rank in range(TOP_N):
        mark = "*" if rank < len(s2_top) and norm(s2_top[rank][0]) in oa_keys else " "
        s2_cell = f"{s2_top[rank][0][:38]} ({s2_top[rank][1]})" if rank < len(s2_top) else ""
        oa_cell = f"{oa_top[rank][0][:38]} ({oa_top[rank][1]})" if rank < len(oa_top) else ""
        print(f"{mark}{s2_cell:<50} | {oa_cell}")

### Result (2026-07-12)

| Seed | top-15 citer overlap |
|---|--:|
| QMIX | **3 / 15** |
| MADDPG | **3 / 15** |
| Rainbow | **6 / 15** |

The *qualitative* story is the finding. For QMIX, S2's top citers are the MARL
landmarks (*StarCraft Multi-Agent Challenge, QTRAN, Value-Decomposition
Networks, MAVEN, QPLEX*); OA's top citers are mostly **applied/journal papers**
(*Distributed Learning in Wireless Networks, Pervasive AI for IoT, Adaptive
Traffic Signal Control, Renewable-energy microgrids*). OA also **duplicates**
records (MuZero appears twice in Rainbow's OA citers) and undercounts the
landmarks' own counts, so its *ranking* is skewed too. So for ML it is **not** a
missing junk tail — the top of the list is a different, lower-quality set, and
because the gap is citation-*extraction*, a preprint-dedup heuristic can't fix
it.

## Conclusion

- **Coverage:** OA undercounts arXiv-native **ML** citations ~3–4×; **physics**
  is fine-to-better. Not a missing physics corpus — a preprint-extraction gap.
- **Resolution:** arXiv-DOI resolves to the preprint stub, and OA offers no
  preprint→VoR link, so an OA-only build needs a canonical-picking heuristic.
- **Quality:** even OA's *top* ML citers differ from S2's landmarks (3–6 / 15
  overlap), so keeping the graph's ML "Field Landmarks" honest needs S2.

Full write-up, the hybrid's division of labor, and the implications for
dropping S2 are in [`docs/citation-coverage.md`](../../docs/citation-coverage.md).

**Caveats:** S2 is the *reference*, not absolute ground truth; the coverage
seeds are famous papers (fame bias); the overlap test is RL-only (only small
citer-count papers are fully pullable from S2).